# CI/CD Best Practices for Snowflake - Demo Session (45 min)

## Agenda
1. **Introduction** - What is CI/CD for Snowflake? (3 min)
2. **Setup** - Git Integration (5 min)
3. **Git Workspaces** - IDE in Snowsight (5 min)
4. **Shared Workspaces** - Team Collaboration (5 min)
5. **DCM Projects** - Infrastructure as Code (12 min)
6. **Pipeline Builder** - Visual Pipeline Design (5 min)
7. **GitHub Actions** - CI/CD Automation (7 min)
8. **Best Practices** - Do's and Don'ts (3 min)

---

## What is CI/CD for Snowflake?

**CI/CD** = Continuous Integration / Continuous Deployment

Traditional approach: Someone runs SQL manually in a worksheet → error-prone, no audit trail, no rollback.

Modern approach with Snowflake DevOps:
- **Define as code** → Declare desired state in version-controlled files
- **Validate before deploy** → Preview changes (plan) before applying them
- **Automate with CI/CD** → Deployments triggered by pull requests and merges, not manual steps

This works with **any Git provider**: GitHub, Azure DevOps, GitLab, Bitbucket.

---
# Section 2: Setup - Connecting Git to Snowflake

To connect a Git repository (GitHub, Azure DevOps, GitLab) to Snowflake, you need 3 objects:

1. **API Integration** - Tells Snowflake which Git providers are allowed
2. **Secret** - Stores authentication credentials (PAT, SSH key, etc.)
3. **Git Repository** - The actual connection to your remote repo

This is a one-time setup. After this, your repo syncs automatically.

In [ ]:
-- Step 1: Create API Integration (tells Snowflake which Git prefixes are allowed)
CREATE OR REPLACE API INTEGRATION CICD_DEMO_GITHUB_API_INTEGRATION
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-rcai')
  ALLOWED_AUTHENTICATION_SECRETS = (SECRETS.PUBLIC.CICD_DEMO_GITHUB_PAT)
  ENABLED = TRUE;

-- Step 2: Create Secret (stores your Personal Access Token)
-- For Azure DevOps: same concept, different PAT source
CREATE OR REPLACE SECRET SECRETS.PUBLIC.CICD_DEMO_GITHUB_PAT
  TYPE = password
  USERNAME = 'sfc-gh-rcai'
  PASSWORD = '<your-github-pat-here>';

-- Step 3: Create Git Repository (connects to the remote repo)
CREATE OR REPLACE GIT REPOSITORY COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO
  API_INTEGRATION = CICD_DEMO_GITHUB_API_INTEGRATION
  GIT_CREDENTIALS = SECRETS.PUBLIC.CICD_DEMO_GITHUB_PAT
  ORIGIN = 'https://github.com/sfc-gh-rcai/snowflake-cicd-demo.git';

In [ ]:
-- Verify: see branches and files from the connected repo
SHOW GIT BRANCHES IN COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO;

In [ ]:
-- List files synced from the repo
LS @COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/dcm/;

---
# Section 3: Git Workspaces

**Switch to Snowsight UI now**

### What to show:
1. Go to **Projects > Workspaces**
2. Open the `snowflake-cicd-demo` workspace (connected to our GitHub repo)
3. Show the file tree - all files from the repo are here
4. Open a SQL file - show editing capabilities
5. Show branching - create a branch, make a change, commit

### Key talking points:
- This is an IDE inside Snowflake connected to your Git repo
- Works with GitHub, Azure DevOps, GitLab - same experience
- Developers can edit, commit, push without leaving Snowflake
- Changes flow back to your CI/CD pipeline automatically
- For Azure DevOps: just use your Azure Repos URL instead of GitHub

---
# Section 4: Shared Workspaces

**Stay in Snowsight UI**

### What to show:
1. Go to **Projects > Workspaces** > Create a new shared workspace
2. Show RBAC: grant access to specific roles
3. Show shared Python utilities (`shared_workspace/utils.py`)
4. Show how notebooks can import from workspace files

### Key talking points:
- **Private workspaces** = your personal dev sandbox (default)
- **Shared workspaces** = team collaboration space (RBAC-governed)
- Store shared Python modules, SQL templates, config files
- Multiple team members can collaborate on the same workspace
- Great for: shared utilities, team libraries, onboarding resources

### Example: Importing shared utils into a notebook
```python
# In a notebook connected to this workspace:
from utils import get_session_info, freshness_check
info = get_session_info(session)
```

---
# Section 5: DCM Projects - Infrastructure as Code

**This is the core of Snowflake DevOps.**

### What is a DCM Project?
- **D**atabase **C**hange **M**anagement
- Declarative: you define the DESIRED STATE, Snowflake figures out what to create/alter/drop
- Like Terraform, but native to Snowflake
- Plan-then-deploy model: always preview before applying

### Project structure:
```
dcm/
├── manifest.yml              ← Targets (DEV, PROD), templating configs
└── sources/
    └── definitions/
        ├── databases.sql     ← DEFINE DATABASE {{ db_name }}
        ├── schemas.sql       ← DEFINE SCHEMA ...
        ├── warehouses.sql    ← DEFINE WAREHOUSE (sized per env)
        ├── roles.sql         ← GRANT statements
        └── pipelines.sql     ← Dynamic Tables pipeline
```

### How Jinja templating works:
Same code deploys to DEV and PROD with different configs:
- DEV: X-SMALL warehouse, 1-hour target lag
- PROD: MEDIUM warehouse, 5-minute target lag

The `manifest.yml` defines these configs. The SQL files use `{{ variables }}`.

In [ ]:
-- Let's look at our manifest.yml (defines environments and configs)
SELECT $1 AS manifest_content
FROM @COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/dcm/manifest.yml;

In [ ]:
-- Let's look at the pipeline definitions (Dynamic Tables with Jinja)
SELECT $1 AS pipeline_definition
FROM @COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/dcm/sources/definitions/pipelines.sql;

### DCM Plan - Preview changes BEFORE deploying

The `PLAN` command compares your definitions against current account state and shows what WOULD change.
Nothing is modified. This is your safety net.

In [ ]:
-- PLAN: Preview what would change for DEV (safe - no modifications)
EXECUTE DCM PROJECT DCM_PROJECTS.PUBLIC.CICD_DEMO_PROJECT_DEV
  PLAN
  USING CONFIGURATION DEV
FROM '@COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/dcm/';

### DCM Deploy - Apply changes

Once you're happy with the plan, deploy to actually create/alter/drop objects.
In CI/CD, `plan` runs on PR (for review), `deploy` runs on merge to main.

In [ ]:
-- DEPLOY: Apply changes for PROD
EXECUTE DCM PROJECT DCM_PROJECTS.PUBLIC.CICD_DEMO_PROJECT_PROD
  DEPLOY
  USING CONFIGURATION PROD
FROM '@COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/dcm/';

In [ ]:
-- Verify: see what was deployed
SHOW DATABASES LIKE 'CICD_DEMO_ANALYTICS%';
SHOW WAREHOUSES LIKE 'CICD_DEMO_WH%';
SHOW DYNAMIC TABLES IN DATABASE CICD_DEMO_ANALYTICS_PROD;

---
# Section 6: Pipeline Builder

**Switch to Snowsight UI**

### What to show:
1. Go to **Data Engineering > Pipeline Builder** (or find it via the Dynamic Tables)
2. Show the visual representation of our pipeline:
   - `RAW.ORDERS` (source table)
   - → `TRANSFORMED.CLEANED_ORDERS` (Dynamic Table - cleans and enriches)
   - → `PRESENTATION.ORDER_SUMMARY` (Dynamic Table - aggregates by region/day)
3. Show how you can visually add new stages, configure target lag, etc.

### Key talking points:
- Pipeline Builder is a visual tool for designing data pipelines
- Under the hood, it creates Dynamic Tables (declarative pipelines)
- The generated code can be exported into your DCM project definitions
- Best of both worlds: visual design for quick iteration, code for CI/CD
- Non-technical users can design, engineers can version-control the output

---
# Section 7: GitHub Actions - CI/CD Automation

## What is GitHub Actions? (and Azure Pipelines)

**GitHub Actions** is GitHub's built-in automation engine. Think of it as a robot that:
1. **Watches** your repo for events (PR opened, code pushed to main)
2. **Spins up** a fresh Linux VM (called a "runner")
3. **Runs** whatever commands you define (install tools, run tests, deploy)
4. **Reports** success/failure back to the PR or commit

**Azure Pipelines** is the exact same concept in Azure DevOps. Same triggers, same YAML-based config, same idea. Just different syntax.

### Our workflow does:
| Event | Job | What it runs |
|-------|-----|-------------|
| PR opened | Plan | `snow dcm plan --target DEV` (preview changes, post for review) |
| Push to main | Deploy | `snow dcm deploy --target DEMO` (apply changes to Snowflake) |

### Authentication:
- **Best practice**: OIDC workload identity (no secrets stored anywhere)
- **Good**: Key-pair auth (private key stored as CI/CD secret)
- **Demo only**: Password (what we're using here for simplicity)

In [ ]:
-- Let's look at our GitHub Actions workflow file
SELECT $1 AS github_actions_workflow
FROM @COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/.github/workflows/snowflake-cicd.yml;

In [ ]:
-- And here's the Azure DevOps equivalent (same commands, different YAML syntax)
SELECT $1 AS azure_pipelines_reference
FROM @COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO/branches/main/azure-pipelines.yml;

### Live Demo: Show GitHub Actions runs

**Open in browser**: https://github.com/sfc-gh-rcai/snowflake-cicd-demo/actions

Show:
1. The workflow runs triggered by our pushes
2. The "Deploy" job succeeding (installs CLI, tests connection, deploys)
3. How a PR would trigger the "Plan" job instead

For Azure DevOps customers: "Your azure-pipelines.yml does the exact same thing using the Snowflake CLI Azure DevOps Extension (Public Preview)"

---
# Section 8: Best Practices - Do's and Don'ts

## DO's
| # | Practice | Why |
|---|----------|-----|
| 1 | Use OIDC / workload identity for CI/CD auth | No long-lived secrets to rotate |
| 2 | Separate DEV / STAGING / PROD environments | Isolate blast radius |
| 3 | Always PLAN before DEPLOY | Catch unintended drops before they execute |
| 4 | Use DCM Projects for declarative state | Idempotent, repeatable, diffable |
| 5 | Version control everything | Audit trail + rollback capability |

## DON'Ts
| # | Anti-Pattern | Why It's Bad |
|---|--------------|--------------|
| 1 | Store credentials in Git | Security risk |
| 2 | Use ACCOUNTADMIN for CI/CD | Overprivileged, no audit granularity |
| 3 | Deploy to PROD without plan | Unreviewed changes break things |
| 4 | Manual deployments alongside CI/CD | Causes drift between Git and actual state |
| 5 | Ignore DCM plan warnings about DROP | Accidental data loss is permanent |

## Rollback Strategy
1. **Git revert + redeploy** → Revert commit, CI/CD auto-deploys previous state
2. **Time Travel** → For accidental data loss, restore from Snowflake Time Travel
3. **DCM plan against old commit** → Checkout previous tag, run plan to see diff

---

## Resources
- [DevOps with Snowflake](https://docs.snowflake.com/en/developer-guide/builders/devops-with-snowflake)
- [DCM Projects Overview](https://docs.snowflake.com/user-guide/dcm-projects/dcm-projects-overview)
- [Snowflake CLI Azure DevOps Extension](https://docs.snowflake.com/en/developer-guide/snowflake-cli/cicd/azure-devops-extension)
- [CI/CD Quickstart](https://www.snowflake.com/en/developers/guides/configure-cicd-integrations-with-snowflake/)
- This repo: https://github.com/sfc-gh-rcai/snowflake-cicd-demo

---
# Cleanup (optional - run after demo if needed)

Run these to remove all demo objects from the account.

In [ ]:
-- CLEANUP: Remove all CICD_DEMO objects (only run if you want to reset)
-- DROP DATABASE IF EXISTS CICD_DEMO_ANALYTICS_DEV;
-- DROP DATABASE IF EXISTS CICD_DEMO_ANALYTICS_PROD;
-- DROP DATABASE IF EXISTS DCM_PROJECTS;
-- DROP WAREHOUSE IF EXISTS CICD_DEMO_WH_DEV;
-- DROP WAREHOUSE IF EXISTS CICD_DEMO_WH_PROD;
-- DROP ROLE IF EXISTS CICD_DEMO_DATA_ENGINEER;
-- DROP ROLE IF EXISTS CICD_DEMO_DATA_ANALYST;
-- DROP ROLE IF EXISTS CICD_DEMO_DEPLOYER;
-- DROP GIT REPOSITORY IF EXISTS COCO_DEMO.PUBLIC.CICD_DEMO_GIT_REPO;
-- DROP SECRET IF EXISTS SECRETS.PUBLIC.CICD_DEMO_GITHUB_PAT;
-- DROP INTEGRATION IF EXISTS CICD_DEMO_GITHUB_API_INTEGRATION;